In [0]:
from pyspark.sql.types import *

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, CATALOG)

In [0]:
TARGET_TABLE_CONF = {
    "table": "pyspark_static_location_lookup",
    "schema": "silver"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
swiss_city_lookup = [
    # city, canton_name, canton_code, region, language_region
    ("Zürich",       "Zürich",                 "ZH", "Zürich",            "German"),
    ("Bern",         "Bern",                   "BE", "Espace Mittelland", "German"),
    ("Basel",        "Basel-Stadt",            "BS", "Nordwestschweiz",   "German"),
    ("Genève",       "Genève",                 "GE", "Lémanique",         "French"),
    ("Lausanne",     "Vaud",                   "VD", "Lémanique",         "French"),
    ("Winterthur",   "Zürich",                 "ZH", "Zürich",            "German"),
    ("St. Gallen",   "St. Gallen",             "SG", "Ostschweiz",        "German"),
    ("Luzern",       "Luzern",                 "LU", "Zentralschweiz",    "German"),
    ("Biel",         "Bern",                   "BE", "Espace Mittelland", "German/French"),
    ("Thun",         "Bern",                   "BE", "Espace Mittelland", "German"),
    ("Aarau",        "Aargau",                 "AG", "Nordwestschweiz",   "German"),
    ("Schaffhausen", "Schaffhausen",           "SH", "Ostschweiz",        "German"),
    ("Lugano",       "Ticino",                 "TI", "Ticino",            "Italian"),
    ("Fribourg",     "Fribourg",               "FR", "Espace Mittelland", "French/German"),
    ("Chur",         "Graubünden",             "GR", "Ostschweiz",        "German"),
    ("Neuchâtel",    "Neuchâtel",              "NE", "Espace Mittelland", "French"),
    ("Zug",          "Zug",                    "ZG", "Zentralschweiz",    "German"),
    ("Sion",         "Valais",                 "VS", "Lémanique",         "French"),
    ("Bellinzona",   "Ticino",                 "TI", "Ticino",            "Italian"),
    ("Liestal",      "Basel-Landschaft",       "BL", "Nordwestschweiz",   "German"),
    ("Solothurn",    "Solothurn",              "SO", "Espace Mittelland", "German"),
    ("Frauenfeld",   "Thurgau",                "TG", "Ostschweiz",        "German"),
    ("Schwyz",       "Schwyz",                 "SZ", "Zentralschweiz",    "German"),
    ("Altdorf",      "Uri",                    "UR", "Zentralschweiz",    "German"),
    ("Sarnen",       "Obwalden",               "OW", "Zentralschweiz",    "German"),
    ("Stans",        "Nidwalden",              "NW", "Zentralschweiz",    "German"),
    ("Glarus",       "Glarus",                 "GL", "Ostschweiz",        "German"),
    ("Herisau",      "Appenzell Ausserrhoden", "AR", "Ostschweiz",        "German"),
    ("Appenzell",    "Appenzell Innerrhoden",  "AI", "Ostschweiz",        "German"),
    ("Delémont",     "Jura",                   "JU", "Espace Mittelland", "French"),
]

city_lookup_schema = StructType([
    StructField("city",            StringType()),
    StructField("canton_name",     StringType()),
    StructField("canton_code",     StringType()),
    StructField("region",          StringType()),
    StructField("language_region", StringType()),
])

df_city_lookup = spark.createDataFrame(swiss_city_lookup, city_lookup_schema)

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"

    (df_city_lookup
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())